# DocShield - Privacy-First Medical Document Assistant

## Gemma 4 Good Hackathon Submission

**Problem:** Billions of people receive medical documents they can't understand — lab reports full of jargon, prescriptions with dangerous drug combinations, hospital bills with hidden overcharges. They upload these sensitive documents to cloud AI tools, giving away their most private data.

**Solution:** DocShield is a multi-agent AI system powered by Gemma 4 that:
- Reads any medical document (photo, scan, or text)
- Explains it in simple language anyone can understand
- Flags dangerous drug interactions
- Catches billing errors and overcharges
- Runs 100% locally — zero data leaves your device

### Gemma 4 Features Used
- **Multimodal Vision** — Reads photos of documents (OCR, handwriting)
- **Native Function Calling** — Queries drug interaction and billing databases
- **Multi-Agent Architecture** — 4 specialized agents working together
- **On-Device** — Designed to run on Gemma 4 E4B for full privacy

## Architecture

```
User uploads document (image or text)
           |
    [Orchestrator Agent]
           |
    [Reader Agent] -- Gemma 4 Vision
           |
    Classifies: prescription / lab_report / hospital_bill
           |
    +------+------+------+
    |             |             |
[Explainer]  [Checker]    [Bill Analyzer]
    |        Uses function    Uses function
    |        calling for      calling for
    |        drug DB          billing DB
```

In [ ]:
# Setup
import sys
import json

# Import DocShield components
from docshield.tools.drug_interactions import check_drug_interaction, DrugInteractionDB
from docshield.tools.billing_reference import lookup_procedure_cost
from docshield.agents.reader_agent import ReaderAgent
from docshield.agents.explainer_agent import ExplainerAgent
from docshield.agents.checker_agent import CheckerAgent
from docshield.agents.bill_analyzer_agent import BillAnalyzerAgent
from docshield.agents.orchestrator import Orchestrator

print('DocShield loaded successfully!')

## Demo 1: Drug Interaction Detection

DocShield's Checker Agent uses Gemma 4's **native function calling** to query a drug interaction database. Here's the tool in action:

In [ ]:
# Demonstrate function calling - Drug Interaction Check
print('=== Drug Interaction Checks ===\n')

# Dangerous combo: blood thinner + pain reliever
result = check_drug_interaction('Warfarin', 'Ibuprofen')
print(f'Warfarin + Ibuprofen:')
print(f'  Severity: {result["severity"].upper()}')
print(f'  Effect: {result["effect"]}')
print(f'  Recommendation: {result["recommendation"]}\n')

# Serotonin syndrome risk
result = check_drug_interaction('Prozac', 'Tramadol')
print(f'Prozac + Tramadol:')
print(f'  Severity: {result["severity"].upper()}')
print(f'  Effect: {result["effect"]}')
print(f'  Recommendation: {result["recommendation"]}\n')

# FDA Black Box Warning
result = check_drug_interaction('Xanax', 'Oxycodone')
print(f'Xanax + Oxycodone:')
print(f'  Severity: {result["severity"].upper()}')
print(f'  Effect: {result["effect"]}')
print(f'  Recommendation: {result["recommendation"]}')

In [ ]:
# Check ALL pairs in a prescription
print('=== Full Prescription Check ===\n')
medications = ['Warfarin', 'Lisinopril', 'Ibuprofen', 'Omeprazole']
print(f'Medications: {medications}\n')

db = DrugInteractionDB()
interactions = db.check_all_pairs(medications)

if interactions:
    for ix in interactions:
        severity_icon = {'high': '!!!', 'medium': '!!', 'low': '!'}
        icon = severity_icon.get(ix['severity'], '?')
        print(f'{icon} {ix["drug_a"]} + {ix["drug_b"]} ({ix["severity"].upper()})')
        print(f'   Effect: {ix["effect"]}')
        print(f'   Action: {ix["recommendation"]}\n')
else:
    print('No interactions found.')

## Demo 2: Hospital Bill Analysis

DocShield's Bill Analyzer uses **function calling** to look up typical costs for medical procedures and flag potential overcharges.

In [ ]:
# Demonstrate billing lookup
print('=== Bill Line Item Analysis ===\n')

bill_items = [
    ('99284', 'ER Visit, High Complexity', 2800),
    ('71046', 'Chest X-ray, 2 views', 950),
    ('93000', 'EKG/ECG', 450),
    ('85025', 'CBC Blood Test', 250),
    ('36415', 'Blood Draw', 95),
]

total_overcharge = 0
for cpt, desc, charged in bill_items:
    result = lookup_procedure_cost(cpt)
    if result['found']:
        typical_max = result['typical_range_usd'][1]
        flag = '  OVERCHARGE' if charged > typical_max else '  OK'
        over = max(0, charged - typical_max)
        total_overcharge += over
        print(f'{desc} (CPT {cpt})')
        print(f'  Charged: ${charged:,.0f} | Typical: ${result["typical_range_usd"][0]}-${typical_max}{flag}')
        if over > 0:
            print(f'  Potential overcharge: ${over:,.0f}')
        print()

print(f'Total potential overcharges: ${total_overcharge:,.0f}')

## Demo 3: Full Pipeline with Gemma 4

Now let's run the complete multi-agent pipeline using Gemma 4. The orchestrator will:
1. Read the document text
2. Classify the document type
3. Route to the appropriate specialist agents

In [ ]:
# Initialize Gemma 4 backend
# On Kaggle: uses google-generativeai SDK
# Locally: uses Ollama
try:
    from docshield.kaggle_backend import KaggleBackend
    backend = KaggleBackend()
    print('Using Kaggle/Gemma 4 backend')
except ImportError:
    from docshield.ollama_backend import OllamaBackend
    backend = OllamaBackend()
    print('Using Ollama backend')

orchestrator = Orchestrator(backend)

In [ ]:
# Run on sample prescription
sample_rx = open('tests/fixtures/sample_prescription.txt').read()
print('INPUT DOCUMENT:')
print(sample_rx)
print('\n' + '='*60 + '\n')
print('DOCSHIELD ANALYSIS:\n')

for event in orchestrator.run({'text': sample_rx}):
    if 'header' in event:
        print(f'\n--- {event["header"]} ---')
    if 'token' in event:
        print(event['token'], end='', flush=True)
    if event.get('done') and event.get('agent'):
        print()

In [ ]:
# Run on sample lab report
sample_lab = open('tests/fixtures/sample_lab_report.txt').read()
print('INPUT DOCUMENT:')
print(sample_lab)
print('\n' + '='*60 + '\n')
print('DOCSHIELD ANALYSIS:\n')

for event in orchestrator.run({'text': sample_lab}):
    if 'header' in event:
        print(f'\n--- {event["header"]} ---')
    if 'token' in event:
        print(event['token'], end='', flush=True)
    if event.get('done') and event.get('agent'):
        print()

In [ ]:
# Run on sample hospital bill
sample_bill = open('tests/fixtures/sample_bill.txt').read()
print('INPUT DOCUMENT:')
print(sample_bill)
print('\n' + '='*60 + '\n')
print('DOCSHIELD ANALYSIS:\n')

for event in orchestrator.run({'text': sample_bill}):
    if 'header' in event:
        print(f'\n--- {event["header"]} ---')
    if 'token' in event:
        print(event['token'], end='', flush=True)
    if event.get('done') and event.get('agent'):
        print()

## Privacy & Impact

### Why Privacy Matters
- Medical documents contain the most sensitive personal data
- People routinely upload these to cloud AI tools with no privacy guarantees
- DocShield processes everything locally — zero data leaves the device
- Gemma 4 E4B enables on-device inference even on smartphones

### Impact
- **Health Literacy:** Billions of people can't understand their medical documents
- **Drug Safety:** Preventable drug interactions cause 125,000+ deaths/year in the US alone
- **Financial Protection:** Medical billing errors affect ~80% of hospital bills
- **Accessibility:** Works in any language, any literacy level, offline

### Gemma 4 Features Showcased
| Feature | How DocShield Uses It |
|---------|----------------------|
| Multimodal Vision | Reads photos of documents, handwritten prescriptions |
| Native Function Calling | Queries drug interaction DB, billing cost DB |
| Multi-Agent | 5 specialized agents with distinct roles |
| On-Device (E4B) | Full privacy, works offline |
| Long Context (256K) | Handles multi-page medical records |